# BERT Classifier Training with LoRA
Multi-head classification: Mental Disease + Severity

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, AdamW, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import json

## Model Definition - Dual Head Classifier

In [ ]:
class DualHeadBERTClassifier(nn.Module):
    def __init__(self, model_name, num_disease_classes, num_severity_classes):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        
        # Disease classification head
        self.disease_classifier = nn.Linear(hidden_size, num_disease_classes)
        
        # Severity classification head
        self.severity_classifier = nn.Linear(hidden_size, num_severity_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        
        disease_logits = self.disease_classifier(pooled_output)
        severity_logits = self.severity_classifier(pooled_output)
        
        return disease_logits, severity_logits

## LoRA Configuration

In [ ]:
# Hyperparameters
MODEL_NAME = "bert-base-uncased"
NUM_DISEASE_CLASSES = 5  # Update based on your dataset
NUM_SEVERITY_CLASSES = 3  # Update based on your dataset
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
NUM_EPOCHS = 3
MAX_LENGTH = 128

# Initialize model
model = DualHeadBERTClassifier(MODEL_NAME, NUM_DISEASE_CLASSES, NUM_SEVERITY_CLASSES)

# LoRA config for efficient fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

# Apply LoRA to BERT backbone only
model.bert = get_peft_model(model.bert, lora_config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded on {device}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

## Data Loading (Placeholder)

In [ ]:
# TODO: Load your dataset here
# Example structure:
# dataset should have: 'text', 'disease_label', 'severity_label'

# train_dataset = ...
# val_dataset = ...

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# TODO: Create DataLoader once dataset is ready
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

## Training Loop

In [ ]:
# Setup optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
# total_steps = len(train_loader) * NUM_EPOCHS
# scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Loss functions
disease_criterion = nn.CrossEntropyLoss()
severity_criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        disease_labels = batch['disease_label'].to(device)
        severity_labels = batch['severity_label'].to(device)
        
        optimizer.zero_grad()
        
        disease_logits, severity_logits = model(input_ids, attention_mask)
        
        loss_disease = disease_criterion(disease_logits, disease_labels)
        loss_severity = severity_criterion(severity_logits, severity_labels)
        loss = loss_disease + loss_severity
        
        loss.backward()
        optimizer.step()
        # scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

# Training loop (commented out until dataset is ready)
# for epoch in range(NUM_EPOCHS):
#     train_loss = train_epoch(model, train_loader, optimizer, device)
#     print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Train Loss: {train_loss:.4f}")

## Save Model Weights

In [ ]:
# Save LoRA adapters and full model state
# model.bert.save_pretrained("./models/lora_adapters")
# torch.save({
#     'model_state_dict': model.state_dict(),
#     'disease_classes': NUM_DISEASE_CLASSES,
#     'severity_classes': NUM_SEVERITY_CLASSES,
# }, "./models/dual_head_classifier.pt")